# 77) İşaret Testi (Sign Test)
İşaret Testi, Non-Parametrik Testler ailesinin **en basit** üyesi — Tek Örneklem T Testi'nin veya Bağımlı Örneklem T Testi'nin normallik varsayımı sağlanmadığında kullanılan alternatifidir.

## Mantığı 
İşaret Testi, verinin **büyüklüğünü tamamen göz ardı eder**, sadece her gözlemin **medyandan (veya karşılaştırma noktasından) büyük mü 
küçük mü** olduğuna (yani "işaretine": + veya -) bakar. Ne kadar büyük ya da küçük olduğu önemli değil, sadece yön önemli.

## Ne Zaman Kullanılır?
- Tek örneklem senaryosu: Bir grubun medyanının, belirli bir değere eşit olup olmadığını test etmek istediğimizde (T testinin ortalama yerine medyan versiyonu gibi düşünülebilir)
- Bağımlı/eşleştirilmiş senaryo: Öncesi-sonrası karşılaştırmasında, fark değerlerinin çoğunlukla pozitif mi negatif mi olduğuna bakmak istediğimizde (Wilcoxon'un çok daha basit/kaba hali — Wilcoxon farkların büyüklüğünü de kullanırken, İşaret Testi sadece yönüne bakar)

## Hipotezler
- **H0:** Medyan, belirli bir değere eşittir (ya da öncesi-sonrası arasında fark yoktur — pozitif ve negatif işaretlerin sayısı eşit 
olmalı, yaklaşık %50-%50)
- **H1:** Medyan farklıdır (pozitif/negatif işaretler dengesiz dağılmıştır)

## Test Mantığı, Aslında Bir Binom Testi!
İşaret Testi, aslında gizlenmiş bir **Binom Dağılımı** uygulamasıdır: Her gözlem ya "+" ya "-" olabilir (Bernoulli/Binom mantığı, tıpkı yazı-tura gibi), ve H0 altında bunun **p=0.5** ile gerçekleşmesi beklenir (yarı yarıya). Test, gözlenen +/- sayısının, bu %50-%50 beklentisinden ne kadar saptığını (Binom testiyle) ölçer.

## Python'da Kullanımı
`scipy`'de doğrudan yoktur, ama Binom testi üzerinden kolayca 
uygulanabilir:
```python
from scipy.stats import binomtest

pozitif_sayisi = 18  # kaç gözlem medyandan/karşılaştırma noktasından büyük
toplam = 25           # toplam gözlem sayısı (eşitlikler genelde çıkarılır)

sonuc = binomtest(k=pozitif_sayisi, n=toplam, p=0.5, alternative='two-sided')
print(sonuc.pvalue)
```

## Dezavantajı
Çok fazla bilgi kaybettiği için (sadece yön, büyüklük yok), Wilcoxon gibi diğer non-parametrik testlerden bile daha **düşük istatistiksel 
güce (Power)** sahiptir — genelde diğer seçenekler tükendiğinde başvurulan bir "son çare" testtir.

In [1]:
import numpy as np
from statsmodels.stats.descriptivestats import sign_test

np.random.seed(42)

oncesi = np.random.normal(300, 50, 20)
sonrasi = oncesi + np.random.normal(15, 40, 20)

# H0: Farkların medyanı sıfıra eşittir, farklar simetrik dağılmıştır.
# H1: Farkların medyanı sıfırdan farklıdır.
farklar = sonrasi - oncesi
print("Farklar:", farklar.round(1))
M, pvalue = sign_test(farklar)
print(f'(+) - (-) / 2 = {M}')
print(f'P-değeri: {pvalue}')
alpha = 0.05
if pvalue < alpha:
    print('H0 reddedilir, farkların medyanı sıfırdan farklı olabilir.')
else:
    print('H0"ı reddecek yeterli kanıt yok, farkların medyanı sıfıra eşit olabilir.')

Farklar: [ 73.6   6.   17.7 -42.   -6.8  19.4 -31.   30.   -9.    3.3  -9.1  89.1
  14.5 -27.3  47.9 -33.8  23.4 -63.4 -38.1  22.9]
(+) - (-) / 2 = 1.0
P-değeri: 0.8238029479980469
H0"ı reddecek yeterli kanıt yok, farkların medyanı sıfıra eşit olabilir.


### Sonuç
Kampanya öncesi ve sonrası müşteri harcamaları arasındaki farkın yönüne (artış/azalış) dair İşaret Testi uyguladık. H0 hipotezine göre, 
pozitif ve negatif işaretli farkların sayısının eşit (her biri n/2, yani 10-10) olması beklenir. 20 müşteriden 11'inin harcaması artmış 
(+), 9'unun azalmış (-) olarak gözlenmiştir — bu, beklenen 10-10 dağılımından çok küçük bir sapmadır. Test sonuçlarına göre bu sapma 
istatistiksel olarak anlamlı bulunmamıştır (p=0.824>0.05), yani kampanyanın harcamalar üzerinde yön bazında anlamlı bir etkisi olduğuna 
dair yeterli kanıt yoktur.